# 3D U-Net on BraTS2020 — runs on Kaggle **and** Colab, resumable on both

**Why this exists.** Our hackathon models are 2D: we segment slice by slice,
because published 3D BraTS models document a 16 GB+ VRAM requirement and the
development laptop has 6 GB. That is a hardware limit, not a method limit, so
"3D would be better" stayed a hypothesis. Both Kaggle and Colab hand out cards
with ~15 GB, which is enough for patch-based 3D — so this turns the hypothesis
into a number.

**Baseline to beat:** mean tumour Dice **0.76**, enhancing tumour **0.84**.

---

## The two platforms, and why use both

|  | Kaggle | Colab |
|---|---|---|
| BraTS data | **already hosted** — mounts instantly | re-download ~14 GB each session |
| Background running | **up to 12 h after closing the tab** (Save & Run All) | free tier stops when idle (~90 min) |
| Interactive idle timeout | ~20 min | ~90 min |
| Checkpoints persist in | `/kaggle/working` (on Save Version) | Google Drive |

**Use Kaggle for long overnight runs, Colab for shorter interactive ones.**
Neither is a second implementation — this is one notebook that detects where it
is running and adapts.

## Moving between them

Checkpoints can be pushed to a **Kaggle Dataset**, which both platforms can read.
So you can train overnight on Kaggle, pull the checkpoint into Colab the next
morning, and continue from the same epoch. Cell 10 does the push; cell 3 pulls.

Every checkpoint carries its optimiser state, epoch counter and full metric
history, so nothing is lost in the handover.

## 1 · Detect the platform

Everything downstream — data paths, where checkpoints live, how to authenticate
— follows from this one cell. Nothing else in the notebook is platform-specific.

In [ ]:
import os, sys, glob, json, random, time

ON_KAGGLE = os.path.exists('/kaggle/input')
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_KAGGLE:
    PLATFORM = 'kaggle'
    # /kaggle/working survives ONLY if you Save a Version. For long runs use
    # "Save & Run All (Commit)" rather than the interactive editor.
    CKPT_DIR = '/kaggle/working/checkpoints'
elif ON_COLAB:
    PLATFORM = 'colab'
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/mri_3d_unet'
else:
    PLATFORM = 'local'
    CKPT_DIR = './checkpoints'

os.makedirs(CKPT_DIR, exist_ok=True)

import torch
print(f"platform    : {PLATFORM}")
print(f"checkpoints : {CKPT_DIR}")
print(f"torch       : {torch.__version__} | CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU         : {p.name}  {p.total_memory/1e9:.1f} GB")
    if p.total_memory / 1e9 < 12:
        print("  ! under 12 GB — set PATCH = 96 in the config cell")
print("existing checkpoints:", sorted(os.listdir(CKPT_DIR)) or "(none — first run)")

## 2 · Get BraTS2020

On **Kaggle**: click *Add Data* in the right-hand panel, search
`brats20-dataset-training-validation` (by awsaf49), add it. It mounts read-only
under `/kaggle/input/` — no download, no wait.

On **Colab**: pulled straight from Kaggle into the VM. Do **not** upload the
dataset from your own machine; it is 14 GB. You need a `kaggle.json` API token
(kaggle.com → Settings → API → Create New Token), which the cell prompts for.

In [ ]:
ROOT = None

if ON_KAGGLE:
    hits = glob.glob('/kaggle/input/**/MICCAI_BraTS2020_TrainingData', recursive=True)
    if hits:
        ROOT = hits[0]
    else:
        raise SystemExit("Add the BraTS2020 dataset via 'Add Data' in the right panel.")
else:
    ROOT = '/content/data/MICCAI_BraTS2020_TrainingData'
    if not os.path.exists(ROOT):
        if not os.path.exists('/root/.kaggle/kaggle.json'):
            from google.colab import files
            print("Upload kaggle.json:")
            files.upload()
            os.makedirs('/root/.kaggle', exist_ok=True)
            os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
            os.chmod('/root/.kaggle/kaggle.json', 0o600)
        os.system('pip install -q kaggle')
        os.system('kaggle datasets download -d awsaf49/'
                  'brats20-dataset-training-validation -p /content --unzip -q')
        os.makedirs('/content/data', exist_ok=True)
        os.system('mv /content/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData '
                  '/content/data/ 2>/dev/null')

cases = sorted(d for d in os.listdir(ROOT) if d.startswith('BraTS20'))
print(f"{len(cases)} cases at {ROOT}")

## 3 · (Optional) Pull checkpoints from the other platform

Skip this if you are continuing on the same platform — cell 7 already finds
local checkpoints.

Use it when **handing a run over**: trained on Kaggle overnight, continuing on
Colab this morning, or the reverse. It pulls the checkpoint dataset you pushed
with cell 10.

Set `CKPT_DATASET` to your own dataset slug once you have created it.

In [ ]:
CKPT_DATASET = ''      # e.g. 'yourname/mri-3d-checkpoints' — leave '' to skip

if CKPT_DATASET:
    if ON_KAGGLE:
        # on Kaggle, add the dataset via "Add Data" instead; then copy it in
        src = glob.glob(f"/kaggle/input/{CKPT_DATASET.split('/')[-1]}/*.pt")
        for f in src:
            os.system(f'cp "{f}" "{CKPT_DIR}/"')
        print(f"copied {len(src)} checkpoint(s) from the attached dataset")
    else:
        os.system('pip install -q kaggle')
        os.system(f'kaggle datasets download -d {CKPT_DATASET} -p {CKPT_DIR} --unzip -q')
        print("pulled:", sorted(os.listdir(CKPT_DIR)))
else:
    print("skipped — using whatever is already in", CKPT_DIR)

## 4 · Configuration

`PATCH = 128` means training on random 128³ crops rather than whole
240×240×155 volumes. Full volumes will not fit, and patches are standard for 3D
medical segmentation — they also act as augmentation, since each epoch sees
different crops.

**Splitting is at patient level, never at patch level.** Patches from one patient
are highly correlated, so splitting below the patient leaks information between
train and validation and inflates every score. Same rule as the 2D work, which
is what makes the comparison fair.

In [ ]:
PATCH       = 128     # 96 if your GPU has under 12 GB
BATCH       = 1       # 3D patches are large; accumulate rather than batch
ACCUM       = 2       # effective batch = BATCH * ACCUM
BASE_FILT   = 16      # 32 roughly doubles memory
LR          = 1e-3
EPOCHS      = 60      # total across ALL sessions, not per session
N_CASES     = 126     # match the 2D run so the comparison is like for like
VAL_FRAC    = 0.2
SEED        = 42

# BraTS labels: 0 bg, 1 necrotic, 2 oedema, 4 enhancing. Label 3 does not exist
# in the raw data, so 4 is remapped to 3 — CrossEntropyLoss needs contiguous
# class indices.
NUM_CLASSES = 4
MODALITIES  = ['t1', 't1ce', 't2', 'flair']

import numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"patch {PATCH}^3 | effective batch {BATCH*ACCUM} | {EPOCHS} epochs total")

## 5 · Dataset

Each item is a random patch from one patient, all four modalities stacked as
channels — the same 4-channel input the 2D model used, for the same reason: each
sequence reveals a different part of the tumour (T1c the enhancing rim, FLAIR/T2
the oedema).

Patches are **biased towards tumour-containing regions**. Without that, most
random 128³ crops from a 240×240×155 volume contain no tumour at all and the
model spends the run learning to predict background.

In [ ]:
import nibabel as nib
from torch.utils.data import Dataset, DataLoader

def norm(v):
    b = v[v > 0]
    if b.size == 0:
        return v.astype(np.float32)
    return np.clip((v - b.mean()) / (b.std() + 1e-8), -5, 5).astype(np.float32)

class BraTS3D(Dataset):
    def __init__(self, case_dirs, patch=PATCH, train=True, samples=2):
        self.dirs, self.patch, self.train = case_dirs, patch, train
        self.samples = samples if train else 1

    def __len__(self):
        return len(self.dirs) * self.samples

    def __getitem__(self, i):
        d = self.dirs[i // self.samples]
        cid = os.path.basename(d)
        vols = [norm(nib.load(f"{d}/{cid}_{m}.nii").get_fdata()) for m in MODALITIES]
        seg = nib.load(f"{d}/{cid}_seg.nii").get_fdata().astype(np.int64)
        seg[seg == 4] = 3
        x = np.stack(vols)
        p = self.patch

        if self.train:
            fg = np.argwhere(seg > 0)
            if len(fg) and np.random.rand() < 0.7:
                c = fg[np.random.randint(len(fg))]
                st = [int(np.clip(c[k] - p // 2, 0, max(seg.shape[k] - p, 0)))
                      for k in range(3)]
            else:
                st = [np.random.randint(0, max(seg.shape[k] - p, 1)) for k in range(3)]
        else:
            st = [max((seg.shape[k] - p) // 2, 0) for k in range(3)]

        sl = tuple(slice(st[k], st[k] + p) for k in range(3))
        x, y = x[(slice(None),) + sl], seg[sl]
        pad = [(0, max(p - x.shape[k + 1], 0)) for k in range(3)]
        if any(b for _a, b in pad):
            x = np.pad(x, [(0, 0)] + pad); y = np.pad(y, pad)
        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy())

dirs = [os.path.join(ROOT, c) for c in cases[:N_CASES]]
dirs = [d for d in dirs if os.path.exists(f"{d}/{os.path.basename(d)}_seg.nii")]
random.Random(SEED).shuffle(dirs)                     # PATIENT-level split
n_val = max(1, int(len(dirs) * VAL_FRAC))
val_dirs, train_dirs = dirs[:n_val], dirs[n_val:]

NW = 2 if ON_KAGGLE else 2
train_dl = DataLoader(BraTS3D(train_dirs, train=True), batch_size=BATCH,
                      shuffle=True, num_workers=NW, pin_memory=True)
val_dl = DataLoader(BraTS3D(val_dirs, train=False), batch_size=1, num_workers=NW)
print(f"{len(train_dirs)} train patients / {len(val_dirs)} val patients "
      f"({len(train_dl)} train patches per epoch)")

## 6 · Model and loss

**Loss = Cross-Entropy + soft Dice**, identical to the 2D run.

Dice is not optional. Background is **99.03 %** of voxels in BraTS, so a model
trained on cross-entropy alone reaches 99 % accuracy by predicting "background"
everywhere and finding no tumour at all. Dice measures region overlap, so that
degenerate solution scores zero.

In [ ]:
import torch.nn as nn, torch.nn.functional as F

def block(i, o):
    return nn.Sequential(
        nn.Conv3d(i, o, 3, padding=1, bias=False), nn.InstanceNorm3d(o), nn.LeakyReLU(0.01, True),
        nn.Conv3d(o, o, 3, padding=1, bias=False), nn.InstanceNorm3d(o), nn.LeakyReLU(0.01, True))

class UNet3D(nn.Module):
    """Same shape as our 2D model: encoder, bottleneck, decoder, skip
    connections. Skips carry fine boundary detail straight across, which keeps
    tumour edges sharp — in medicine the boundary is the finding."""
    def __init__(self, in_ch=4, n_cls=NUM_CLASSES, f=BASE_FILT):
        super().__init__()
        self.e1, self.e2, self.e3 = block(in_ch, f), block(f, f*2), block(f*2, f*4)
        self.bott = block(f*4, f*8)
        self.u3 = nn.ConvTranspose3d(f*8, f*4, 2, 2); self.d3 = block(f*8, f*4)
        self.u2 = nn.ConvTranspose3d(f*4, f*2, 2, 2); self.d2 = block(f*4, f*2)
        self.u1 = nn.ConvTranspose3d(f*2, f,   2, 2); self.d1 = block(f*2, f)
        self.out = nn.Conv3d(f, n_cls, 1)
        self.pool = nn.MaxPool3d(2)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        b  = self.bott(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b),  e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)                            # raw logits

def dice_loss(logits, target, eps=1.0):
    p = F.softmax(logits, 1)
    t = F.one_hot(target, NUM_CLASSES).permute(0, 4, 1, 2, 3).float()
    dims = (0, 2, 3, 4)
    inter = (p * t).sum(dims); denom = p.sum(dims) + t.sum(dims)
    return 1 - ((2 * inter + eps) / (denom + eps))[1:].mean()   # ignore background

ce = nn.CrossEntropyLoss()
def criterion(lg, t): return ce(lg, t) + dice_loss(lg, t)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = UNet3D().to(dev)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters "
      f"(the 2D model has 7.77 M)")

## 7 · Resume

Finds the newest checkpoint and restores weights, optimiser state, epoch counter
and metric history. Works the same whether the checkpoint came from this
platform or was pulled from the other one in cell 3.

First run: "starting fresh". Every run after: the epoch it continues from.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler()
start_epoch, history, best_dice = 0, [], 0.0

found = sorted(glob.glob(f"{CKPT_DIR}/epoch_*.pt")) or \
        ([f"{CKPT_DIR}/best.pt"] if os.path.exists(f"{CKPT_DIR}/best.pt") else [])
if found:
    ck = torch.load(found[-1], map_location=dev)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    if 'scaler' in ck: scaler.load_state_dict(ck['scaler'])
    start_epoch = ck['epoch'] + 1
    history     = ck.get('history', [])
    best_dice   = ck.get('best_dice', 0.0)
    print(f"RESUMING from {os.path.basename(found[-1])} -> epoch {start_epoch}")
    print(f"best mean tumour Dice so far: {best_dice:.4f}")
    print(f"trained on: {ck.get('platform', 'unknown')}")
else:
    print("starting fresh (no checkpoint found)")
print(f"will train epochs {start_epoch} -> {EPOCHS-1}")

## 8 · Train

Safe to interrupt at any point — the previous epoch is already checkpointed.

**On Kaggle, run this via "Save & Run All (Commit)"** so it keeps going for up
to 12 hours after you close the tab. The interactive editor idles out in about
20 minutes.

Per-class Dice is reported every epoch, so you can watch the enhancing-tumour
class specifically — clinically the most important, and the one our 2D model
scores best on (0.84).

In [ ]:
CLASSES = ['necrotic', 'oedema', 'enhancing']

@torch.no_grad()
def validate():
    model.eval()
    inter = np.zeros(NUM_CLASSES); denom = np.zeros(NUM_CLASSES); losses = []
    for x, y in val_dl:
        x, y = x.to(dev), y.to(dev)
        with torch.cuda.amp.autocast():
            lg = model(x); losses.append(criterion(lg, y).item())
        pr = lg.argmax(1)
        for c in range(NUM_CLASSES):
            pc, tc = (pr == c), (y == c)
            inter[c] += (pc & tc).sum().item()
            denom[c] += pc.sum().item() + tc.sum().item()
    return float(np.mean(losses)), np.where(denom > 0, 2*inter/np.maximum(denom, 1), np.nan)

for ep in range(start_epoch, EPOCHS):
    model.train(); t0 = time.time(); tot = 0.0
    opt.zero_grad(set_to_none=True)
    for i, (x, y) in enumerate(train_dl):
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        with torch.cuda.amp.autocast():
            loss = criterion(model(x), y) / ACCUM
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM == 0:
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        tot += loss.item() * ACCUM

    tr = tot / max(len(train_dl), 1)
    vl, dice = validate()
    mean_tumour = float(np.nanmean(dice[1:]))
    history.append({'epoch': ep, 'train_loss': tr, 'val_loss': vl,
                    'dice': [None if np.isnan(d) else float(d) for d in dice],
                    'mean_tumour_dice': mean_tumour, 'platform': PLATFORM})

    per = "  ".join(f"{n} {dice[i+1]:.3f}" for i, n in enumerate(CLASSES))
    print(f"epoch {ep:3d} | train {tr:.4f} | val {vl:.4f} | "
          f"mean tumour Dice {mean_tumour:.4f} | {per} | {time.time()-t0:.0f}s", flush=True)

    ck = {'model': model.state_dict(), 'opt': opt.state_dict(),
          'scaler': scaler.state_dict(), 'epoch': ep, 'history': history,
          'best_dice': max(best_dice, mean_tumour), 'platform': PLATFORM,
          'config': {'patch': PATCH, 'base_filters': BASE_FILT,
                     'n_cases': N_CASES, 'seed': SEED}}
    torch.save(ck, f"{CKPT_DIR}/epoch_{ep:03d}.pt")
    if mean_tumour > best_dice:
        best_dice = mean_tumour
        torch.save(ck, f"{CKPT_DIR}/best.pt")
        print(f"   new best ({best_dice:.4f}) -> best.pt", flush=True)

    for f in sorted(glob.glob(f"{CKPT_DIR}/epoch_*.pt"))[:-3]:   # keep 3 newest
        os.remove(f)
    with open(f"{CKPT_DIR}/history.json", 'w') as f:
        json.dump(history, f, indent=2)

print(f"\ndone through epoch {EPOCHS-1}. best mean tumour Dice: {best_dice:.4f}")

## 9 · Did 3D beat 2D?

The comparison this notebook exists to make. Same dataset, same patient-level
split rule, same loss, same metric.

**One caveat, stated rather than buried.** Our 2D number is computed over whole
validation slices; this validates on centre patches. Close, but not an identical
evaluation — so treat a fraction of a point as noise. A real win should be
several points, and **if 3D does not clearly win, that is a legitimate result
worth reporting.**

In [ ]:
import matplotlib.pyplot as plt

hist = json.load(open(f"{CKPT_DIR}/history.json"))
ep = [h['epoch'] for h in hist]

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))
a.plot(ep, [h['train_loss'] for h in hist], label='train')
a.plot(ep, [h['val_loss'] for h in hist], label='validation')
a.set_xlabel('epoch'); a.set_ylabel('loss'); a.legend(); a.grid(alpha=.3)
a.set_title('Learning curve — a growing gap means overfitting')

b.plot(ep, [h['mean_tumour_dice'] for h in hist], lw=2, label='3D U-Net (this run)')
b.axhline(0.76, ls='--', c='#b4432c', label='our 2D baseline (0.76)')
for i, n in enumerate(CLASSES):
    b.plot(ep, [h['dice'][i+1] for h in hist], alpha=.5, lw=1, label=n)
b.set_xlabel('epoch'); b.set_ylabel('Dice'); b.legend(fontsize=8); b.grid(alpha=.3)
b.set_title('Tumour Dice against the 2D result')
plt.tight_layout(); plt.savefig(f"{CKPT_DIR}/curves.png", dpi=130); plt.show()

best = max(h['mean_tumour_dice'] for h in hist)
d = best - 0.76
print(f"best 3D mean tumour Dice : {best:.4f}")
print(f"our 2D baseline          : 0.7600")
print(f"difference               : {d:+.4f}  "
      f"({'3D wins' if d > 0.02 else 'no clear win — report it as such'})")
print(f"epochs completed         : {len(hist)} / {EPOCHS}")
print("platforms used           :", sorted({h.get('platform','?') for h in hist}))

## 10 · Push checkpoints so the other platform can continue

This is the handover. It publishes `best.pt` and `history.json` as a **Kaggle
Dataset**, which both Kaggle and Colab can read — so a run started on one can be
finished on the other.

Needs a `kaggle.json` token. On Kaggle, store it via *Add-ons → Secrets*; on
Colab the cell prompts for the file.

Run this at the **end of a session**, not every epoch — each push creates a new
dataset version.

In [ ]:
DATASET_SLUG = ''      # e.g. 'yourname/mri-3d-checkpoints'

if not DATASET_SLUG:
    print("set DATASET_SLUG to push. Skipping.")
else:
    os.system('pip install -q kaggle')
    stage = '/tmp/ckpt_push'; os.makedirs(stage, exist_ok=True)
    for f in ('best.pt', 'history.json', 'curves.png'):
        if os.path.exists(f"{CKPT_DIR}/{f}"):
            os.system(f'cp "{CKPT_DIR}/{f}" "{stage}/"')

    user, name = DATASET_SLUG.split('/')
    meta = {"title": name, "id": DATASET_SLUG,
            "licenses": [{"name": "CC0-1.0"}]}
    with open(f"{stage}/dataset-metadata.json", 'w') as f:
        json.dump(meta, f)

    # first push creates it; later pushes add a version
    rc = os.system(f'kaggle datasets create -p {stage} -q 2>/dev/null')
    if rc != 0:
        last = json.load(open(f"{CKPT_DIR}/history.json"))[-1]
        os.system(f'kaggle datasets version -p {stage} '
                  f'-m "epoch {last["epoch"]}, dice {last["mean_tumour_dice"]:.4f}" -q')
    print(f"pushed to https://kaggle.com/datasets/{DATASET_SLUG}")
    print("To continue elsewhere: set CKPT_DATASET in cell 3 to this slug.")